In [17]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import numpy as np
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel

PARTICIPANTS = ["jf", "kr", "nh"]

# ---------------------------------------------------------------------
# 1. Load and prep data - identical to the existing rr98 pipeline
# ---------------------------------------------------------------------

df = pd.read_csv("../rr98.csv")
df = df[df["outlier"] == False].copy()
df["correct"] = df["correct"].astype(int)
df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})
assert df["sat_id"].notna().all()

def _qcut_levels(s, n_levels=7):
    return pd.qcut(s, q=n_levels, labels=False, duplicates="drop") + 1

df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
assert df["diff_level"].notna().all()

df["cell"] = (df["sat_id"] - 1) * 7 + df["diff_level"]


def build_data(pid):
    d = df[df["id"] == pid]
    d_correct = d[d["correct"] == 1]
    d_false = d[d["correct"] == 0]

    max_rt = float(d["rt"].max())
    t0_hi = float(d["rt"].quantile(0.05))  # scalar - t0 doesn't vary here

    return {
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(), "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "max_rt": max_rt, "t0_hi": t0_hi,
    }


def init_ddm(data):
    return {
        "a": [2.10, 2.16], "v_base": [0.8]*7,
        "sv": 0.1, "sz": 0.05,
        "t0": 0.2 * data["t0_hi"], "p_lapse": 0.02,
    }


# ---------------------------------------------------------------------
# 2. Fit - sequential across participants (this model is much cheaper
#    per-trial than the quadrature version, since sv is now analytic
#    rather than quadrature-integrated - but still test one participant's
#    runtime before assuming all three will be fast)
# ---------------------------------------------------------------------

model = CmdStanModel(stan_file="DDM_rr98_analytic_sv_3pt.stan")

os.makedirs("ddm_analytic_fits/idata", exist_ok=True)
summaries = []
diagnostics = []

for pid in PARTICIPANTS:
    print(f"=== Fitting analytic-sv DDM: {pid} ===")
    data = build_data(pid)
    inits = init_ddm(data)

    fit = model.sample(
        data=data, chains=4, parallel_chains=2,
        inits=inits, iter_warmup=500, iter_sampling=800,
        adapt_delta=0.95, show_progress=True, show_console=True
    )

    summary = fit.summary().reset_index().rename(columns={"index": "param"})
    summary["participant_id"] = pid
    summaries.append(summary)

    sv_diag = fit.method_variables()
    diagnostics.append({
        "participant_id": pid,
        "n_divergent": int(sv_diag["divergent__"].sum()),
        "pct_divergent": float(sv_diag["divergent__"].sum()) / sv_diag["divergent__"].size,
        "mean_treedepth": float(np.mean(sv_diag["treedepth__"])),
        "max_rhat": float(summary["R_hat"].max()),
        "min_ess_bulk": float(summary["ESS_bulk"].min()),
    })

    idata = az.from_cmdstanpy(fit, log_likelihood="log_lik")
    idata.to_netcdf(f"ddm_analytic_fits/idata/ddm_analytic_{pid}.nc")

pd.concat(summaries).to_csv("ddm_analytic_fits/ddm_analytic_summaries.csv", index=False)
pd.DataFrame(diagnostics).to_csv("ddm_analytic_fits/ddm_analytic_diagnostics.csv", index=False)
print(pd.DataFrame(diagnostics))

16:30:34 - cmdstanpy - INFO - compiling stan file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_3pt.stan to exe file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_3pt
16:30:44 - cmdstanpy - INFO - compiled model executable: /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_3pt


=== Fitting analytic-sv DDM: jf ===


16:30:44 - cmdstanpy - INFO - Chain [1] start processing
16:30:44 - cmdstanpy - INFO - Chain [2] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 800
Chain [1] num_warmup = 500
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpvdkmd3mn/a4qf1r3z.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000g

17:58:59 - cmdstanpy - INFO - Chain [2] done processing
17:58:59 - cmdstanpy - INFO - Chain [3] start processing


...

Chain [2] 
Chain [2] Elapsed Time: 9263.94 seconds (Warm-up)
Chain [2] 1936.5 seconds (Sampling)
Chain [2] 11200.4 seconds (Total)
Chain [2] 
Chain [2] 
Chain [4] method = sample (Default)
Chain [4] sample
Chain [4] num_samples = 800
Chain [4] num_warmup = 500
Chain [4] save_warmup = false (Default)
Chain [4] thin = 1 (Default)
Chain [4] adapt
Chain [4] engaged = true (Default)
Chain [4] gamma = 0.05 (Default)
Chain [4] delta = 0.95
Chain [4] kappa = 0.75 (Default)
Chain [4] t0 = 10 (Default)
Chain [4] init_buffer = 75 (Default)
Chain [4] term_buffer = 50 (Default)
Chain [4] window = 25 (Default)
Chain [4] save_metric = false (Default)
Chain [4] algorithm = hmc (Default)
Chain [4] hmc
Chain [4] engine = nuts (Default)
Chain [4] nuts
Chain [4] max_depth = 10 (Default)
Chain [4] metric = diag_e (Default)
Chain [4] metric_file =  (Default)
Chain [4] stepsize = 1 (Default)
Chain [4] stepsize_jitter = 0 (Default)
Chain [4] num_chains = 1 (Default)
Chain [4] id = 4
Chain [4] data
Chain [4] f

01:50:32 - cmdstanpy - INFO - Chain [3] done processing


Chain [3] 
Chain [3] Elapsed Time: 1521.96 seconds (Warm-up)
Chain [3] 3800.4 seconds (Sampling)
Chain [3] 5322.36 seconds (Total)
Chain [3] 
Chain [3] 
Chain [4] Iteration: 1200 / 1300 [ 92%]  (Sampling)
Chain [4] Iteration: 1300 / 1300 [100%]  (Sampling)


01:54:35 - cmdstanpy - INFO - Chain [4] done processing
01:54:35 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: log_mix: lambda2 is nan, but must be not nan! (in 'DDM_rr98_analytic_sv_3pt.stan', line 113, column 6 to line 116, column 8)
Exception: log_mix: lambda2 is nan, but must be not nan! (in 'DDM_rr98_analytic_sv_3pt.stan', line 113, column 6 to line 116, column 8)
Exception: log_mix: lambda2 is nan, but must be not nan! (in 'DDM_rr98_analytic_sv_3pt.stan', line 113, column 6 to line 116, column 8)
Exception: log_mix: lambda2 is nan, but must be not nan! (in 'DDM_rr98_analytic_sv_3pt.stan', line 113, column 6 to line 116, column 8)


Chain [4] 
Chain [4] Elapsed Time: 1762.22 seconds (Warm-up)
Chain [4] 3749.25 seconds (Sampling)
Chain [4] 5511.47 seconds (Total)
Chain [4] 
Chain [4] 
  participant_id  n_divergent  pct_divergent  mean_treedepth  max_rhat  \
0             jf            0            0.0        3.894375   1.00713   
1             kr            0            0.0        3.877500   1.00421   
2             nh            0            0.0        3.982500   1.00453   

   min_ess_bulk  
0       793.770  
1      1210.790  
2       790.838  


In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import numpy as np
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel

PARTICIPANT = "jf"  # pick whichever one you want to test

df = pd.read_csv("../rr98.csv")
df = df[df["outlier"] == False].copy()
df["correct"] = df["correct"].astype(int)
df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

def _qcut_levels(s, n_levels=7):
    return pd.qcut(s, q=n_levels, labels=False, duplicates="drop") + 1

df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
df["cell"] = (df["sat_id"] - 1) * 7 + df["diff_level"]

def build_data(pid):
    d = df[df["id"] == pid]
    d_correct = d[d["correct"] == 1]
    d_false = d[d["correct"] == 0]
    max_rt = float(d["rt"].max())
    t0_hi = float(d["rt"].quantile(0.05))
    return {
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(), "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "max_rt": max_rt, "t0_hi": t0_hi,
    }

def init_ddm(data):
    return {
        "a": [2.10, 2.16], "v_base": [0.8]*7,
        "sv": 0.1, "sz": 0.05,
        "t0": 0.2 * data["t0_hi"], "p_lapse": 0.02,
    }

model = CmdStanModel(stan_file="DDM_rr98_analytic_sv_3pt.stan")
os.makedirs("ddm_analytic_fits/idata", exist_ok=True)

data = build_data(PARTICIPANT)
inits = init_ddm(data)

fit = model.sample(
    data=data, chains=4, parallel_chains=4,   # all 4 at once, no other participant to share cores with
    inits=inits, iter_warmup=300, iter_sampling=300,
    adapt_delta=0.95, show_progress=True, show_console=True,
)

summary = fit.summary().reset_index().rename(columns={"index": "param"})
summary["participant_id"] = PARTICIPANT
summary.to_csv(f"ddm_analytic_fits/summary_{PARTICIPANT}.csv", index=False)

sv_diag = fit.method_variables()
print("pct_divergent:", float(sv_diag["divergent__"].sum()) / sv_diag["divergent__"].size)
print("mean_treedepth:", float(np.mean(sv_diag["treedepth__"])))
print("max_rhat:", float(summary["R_hat"].max()))
print("min_ess_bulk:", float(summary["ESS_bulk"].min()))

idata = az.from_cmdstanpy(fit, log_likelihood="log_lik")
idata.to_netcdf(f"ddm_analytic_fits/idata/ddm_analytic_{PARTICIPANT}.nc")

01:55:39 - cmdstanpy - INFO - Chain [1] start processing
01:55:39 - cmdstanpy - INFO - Chain [2] start processing
01:55:39 - cmdstanpy - INFO - Chain [3] start processing
01:55:39 - cmdstanpy - INFO - Chain [4] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 300
Chain [1] num_warmup = 300
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpvdkmd3mn/f6ki7hju.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000g

In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import numpy as np
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel

PARTICIPANT = "jf"  # pick whichever one you want to test

df = pd.read_csv("../rr98.csv")
df = df[df["outlier"] == False].copy()
df["correct"] = df["correct"].astype(int)
df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

def _qcut_levels(s, n_levels=7):
    return pd.qcut(s, q=n_levels, labels=False, duplicates="drop") + 1

df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
df["cell"] = (df["sat_id"] - 1) * 7 + df["diff_level"]

def build_data(pid):
    d = df[df["id"] == pid]
    d_correct = d[d["correct"] == 1]
    d_false = d[d["correct"] == 0]
    max_rt = float(d["rt"].max())
    t0_hi = float(d["rt"].quantile(0.05))
    return {
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(), "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "max_rt": max_rt, "t0_hi": t0_hi,
    }

def init_ddm(data):
    return {
        "a": [2.10, 2.16], "v_base": [0.8]*7,
        "sv": 0.1, "sz": 0.05,
        "t0": 0.2 * data["t0_hi"], "p_lapse": 0.02,
    }

model = CmdStanModel(stan_file="DDM_rr98_analytic_sv_vectorized.stan")
os.makedirs("ddm_analytic_fits/idata", exist_ok=True)

data = build_data(PARTICIPANT)
inits = init_ddm(data)

fit = model.sample(
    data=data, chains=4, parallel_chains=4,   # all 4 at once, no other participant to share cores with
    inits=inits, iter_warmup=300, iter_sampling=300,
    adapt_delta=0.95, show_progress=True, show_console=True,
)

summary = fit.summary().reset_index().rename(columns={"index": "param"})
summary["participant_id"] = PARTICIPANT
summary.to_csv(f"ddm_analytic_fits/summary_{PARTICIPANT}.csv", index=False)

sv_diag = fit.method_variables()
print("pct_divergent:", float(sv_diag["divergent__"].sum()) / sv_diag["divergent__"].size)
print("mean_treedepth:", float(np.mean(sv_diag["treedepth__"])))
print("max_rhat:", float(summary["R_hat"].max()))
print("min_ess_bulk:", float(summary["ESS_bulk"].min()))

idata = az.from_cmdstanpy(fit, log_likelihood="log_lik")
idata.to_netcdf(f"ddm_analytic_fits/idata/ddm_analytic_{PARTICIPANT}.nc")

15:46:00 - cmdstanpy - INFO - compiling stan file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_vectorized.stan to exe file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_vectorized
15:46:12 - cmdstanpy - INFO - compiled model executable: /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/DDM_rr98_analytic_sv_vectorized
15:46:12 - cmdstanpy - INFO - Chain [1] start processing
15:46:12 - cmdstanpy - INFO - Chain [2] start processing
15:46:12 - cmdstanpy - INFO - Chain [3] start processing
15:46:12 - cmdstanpy - INFO - Chain [4] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 300
Chain [1] num_warmup = 300
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/tmpvdkmd3mn/z8jedfim.json
Chain [1] init = /var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000g

15:46:25 - cmdstanpy - INFO - Chain [3] done processing


Chain [1] Chain [3] 
Chain [4] 
Chain [2] 



15:46:25 - cmdstanpy - INFO - Chain [4] done processing
15:46:25 - cmdstanpy - INFO - Chain [1] done processing
15:46:25 - cmdstanpy - INFO - Chain [2] done processing


KeyboardInterrupt: 

15:46:25 - cmdstanpy - ERROR - Chain [3] error: terminated by signal 2 Unknown error: -2
15:46:25 - cmdstanpy - ERROR - Chain [4] error: terminated by signal 2 Unknown error: -2
15:46:25 - cmdstanpy - ERROR - Chain [1] error: terminated by signal 2 Unknown error: -2
15:46:25 - cmdstanpy - ERROR - Chain [2] error: terminated by signal 2 Unknown error: -2
